In [1]:
import pandas as pd
import numpy as np

# ==============================================================================
# EXPERIMENT: THE STRING POINTER TRAP vs CATEGORICAL vs PYARROW
# ==============================================================================

N = 10_000_000

# Generating 1 Crore rows of text data
data_strings = np.random.choice(["Male", "Female"], size=N)

# 1. The Traditional Object Pointer Trap
s_object = pd.Series(data_strings, dtype="object")
mem_object = s_object.memory_usage(deep=True) / (1024 * 1024)
print(f"1. Object Dtype Memory (Pointers + Scattered Text) : {mem_object:.2f} MB")

# 2. The Modern PyArrow String Backend
# Requires PyArrow installed (pip install pyarrow)
try:
    s_arrow = pd.Series(data_strings, dtype="string[pyarrow]")
    mem_arrow = s_arrow.memory_usage(deep=True) / (1024 * 1024)
    print(f"2. PyArrow Dtype Memory (Continuous Bytes + Offsets) : {mem_arrow:.2f} MB")
except Exception as e:
    print("PyArrow not installed or supported in this version.")

# 3. The Categorical Encoding Hack
s_category = pd.Series(data_strings, dtype="category")
mem_category = s_category.memory_usage(deep=True) / (1024 * 1024)
print(f"3. Category Dtype Memory (Dictionary Mapping + Int8) : {mem_category:.2f} MB")

# Comparing the optimization
print("-" * 50)
print(f"Memory Saved via Categorical: {((mem_object - mem_category) / mem_object) * 100:.2f}%")

1. Object Dtype Memory (Pointers + Scattered Text) : 514.99 MB
2. PyArrow Dtype Memory (Continuous Bytes + Offsets) : 123.98 MB
3. Category Dtype Memory (Dictionary Mapping + Int8) : 9.54 MB
--------------------------------------------------
Memory Saved via Categorical: 98.15%
